# BiVLA — UniVLA + AutoGaze on SimplerEnv (Colab Pro)

멘토님 **BiVLA 모노레포**(`SoumyaratnaDebnath/BiVLA`)의 **UniVLA 경로**(adaptive_sparse_vla + AutoGaze + shared controller)를
SimplerEnv(WidowX Bridge)로 돌리는 노트북.

### SpatialVLA 노트북과의 차이 (중요)
- **conda env가 다름: `bivla`** (SpatialVLA는 `spatialvla`). 의존성 충돌 때문에 **반드시 분리**.
- **transformers 4.44.0** (Emu3 백본용). SpatialVLA의 4.47.0과 충돌하므로 별도 env.
- **SimplerEnv는 레포 번들**(`BiVLA/SimplerEnv`)을 그대로 씀 — DelinQu fork 클론 불필요.
- **레포 클론은 동일**: 같은 BiVLA 모노레포를 클론 → `shared_unified_policy.py`(컨트롤러)를 SpatialVLA와 공유.
- 모델: `BAAI/Emu3-VisionTokenizer` + `Yuqi1997/UniVLA`(정책 ~14GB). AutoGaze는 `nvidia/AutoGaze`가 첫 실행 시 자동 다운로드.

### GPU
**A100 / L4 권장** (Emu3 14GB + AutoGaze + 렌더링). Colab Pro에서 런타임 → 고용량 GPU 선택.

> `%%bash` 셀은 bash, 나머지는 Python 셀.


## 1. Miniconda 설치

In [ ]:
%%bash
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
bash /tmp/miniconda.sh -b -f -p /usr/local
conda --version

## 2. conda TOS 동의

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

## 3. conda 환경 생성 (bivla, python 3.10)

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda create -n bivla python=3.10.12 -y
conda run -n bivla python --version

## 4. 시스템 렌더링 의존성 (EGL / Vulkan / ffmpeg)

In [ ]:
%%bash
apt-get update -yqq
apt-get -yqq install libegl1-mesa libegl1 libgl1 libosmesa6-dev
apt-get install -yqq --no-install-recommends libvulkan-dev vulkan-tools
apt-get install -yqq ffmpeg

## 5. 컴파일러 + 파이썬 패키지 (Emu3 / AutoGaze 용)

**transformers 4.44.0** (Emu3 필수), **tiktoken**(Emu3 토크나이저), **einops**(AutoGaze), safetensors/accelerate.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda run -n bivla conda install -c conda-forge gcc=12.1.0 gxx_linux-64 -y
conda run -n bivla pip install mediapy matplotlib
# UniVLA는 Emu3 백본 → transformers 4.44.0
conda run -n bivla pip install transformers==4.44.0 tokenizers==0.19.1 pillow
# Emu3 토크나이저 + AutoGaze 의존성
conda run -n bivla pip install tiktoken einops timm safetensors accelerate huggingface_hub transforms3d scipy

## 6. BiVLA 모노레포 클론 + 번들 SimplerEnv 설치

**SpatialVLA와 같은 레포를 클론**한다(컨트롤러 공유). SimplerEnv는 **번들된 `BiVLA/SimplerEnv`** 를 그대로 설치.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh

# 멘토님 BiVLA 모노레포 (SpatialVLA 노트북과 동일 레포)
if [ ! -d /content/BiVLA ]; then
  git clone --depth 1 https://github.com/SoumyaratnaDebnath/BiVLA.git /content/BiVLA
fi

SIM=/content/BiVLA/SimplerEnv

# SAPIEN 2.2.2
conda run -n bivla pip install sapien==2.2.2

# ManiSkill2_real2sim (번들)
conda run -n bivla pip install --no-deps -e $SIM/ManiSkill2_real2sim

# gym
conda run -n bivla conda install -c conda-forge gym=0.21.0 -y

# ruckig
conda run -n bivla pip install ruckig

# 시뮬 의존성
conda run -n bivla pip install \
  transforms3d "opencv-python-headless==4.8.1.78" \
  "trimesh==3.22.5" "open3d==0.17.0" "mplib==0.0.9" "gymnasium==0.29.1"

# mani-skill2 누락 deps
conda run -n bivla pip install \
  gdown GitPython h5py \
  imageio "imageio[ffmpeg]" \
  rtree tabulate

# numpy 고정
conda run -n bivla pip install "numpy==1.24.4"

# SimplerEnv (번들)
conda run -n bivla pip install --no-deps -e $SIM

## 7. opencv / numpy 버전 재고정

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda run -n bivla pip install "opencv-python==4.8.1.78"
conda run -n bivla pip install "numpy==1.24.4" 

## 8. xvfb (헤드리스 디스플레이)

In [ ]:
%%bash
apt-get install -yqq xvfb
echo "xvfb done" 

## 9. Vulkan ICD 설정

In [ ]:
%%bash
mkdir -p /etc/vulkan/icd.d
cat > /etc/vulkan/icd.d/nvidia_icd.json << 'EOF'
{
    "file_format_version": "1.0.0",
    "ICD": {
        "library_path": "libGLX_nvidia.so.0",
        "api_version": "1.2.155"
    }
}
EOF
echo "Vulkan config done" 

## 10. pkg_resources shim (setuptools 충돌 회피)

In [ ]:
import os, subprocess
site = "/usr/local/envs/bivla/lib/python3.10/site-packages"
pr = os.path.join(site, "pkg_resources")
ok = False
if os.path.exists(os.path.join(pr, "__init__.py")):
    out = subprocess.run(["/usr/local/envs/bivla/bin/python","-c","import pkg_resources;print('ok')"],
                         capture_output=True, text=True)
    ok = "ok" in out.stdout
if ok:
    print("pkg_resources 정상 — shim 불필요")
else:
    os.makedirs(pr, exist_ok=True)
    shim = '''import importlib, importlib.metadata as _meta, os
def get_distribution(name):
    class _D:
        try: version=_meta.version(name)
        except Exception: version="0.0.0"
    return _D()
def require(r): pass
def resource_filename(p,r):
    try:
        m=importlib.import_module(p); return os.path.join(os.path.dirname(m.__file__),r)
    except Exception: return r
def resource_string(p,r):
    with open(resource_filename(p,r),"rb") as f: return f.read()
def resource_exists(p,r): return os.path.exists(resource_filename(p,r))
def resource_stream(p,r): return open(resource_filename(p,r),"rb")
class WorkingSet:
    def __iter__(self): return iter([])
    def __contains__(self,i): return False
    def require(self,*a,**k): pass
working_set=WorkingSet()
'''
    with open(os.path.join(pr,"__init__.py"),"w") as f: f.write(shim)
    print("pkg_resources shim 생성:", pr)

## 11. torch CUDA 12.1 설치

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda run -n bivla pip install torch==2.5.1 torchvision==0.20.1 \
  --index-url https://download.pytorch.org/whl/cu121 -q
conda run -n bivla python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 12. 시뮬레이터 + Emu3 + AutoGaze import 검증

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
export PYTHONPATH=/content/BiVLA:/content/BiVLA/UniVLA:/content/BiVLA/UniVLA/reference/Emu3:/content/BiVLA/AutoGaze:/content/BiVLA/SimplerEnv:$PYTHONPATH
conda run -n bivla python -c "import sapien.core; print('sapien OK')"
conda run -n bivla python -c "import simpler_env; print('simpler_env OK')"
conda run -n bivla python -c "import sys; sys.path.insert(0,'/content/BiVLA/UniVLA/reference/Emu3'); from emu3.mllm import Emu3Tokenizer; print('emu3 OK')"
conda run -n bivla python -c "import sys; sys.path.insert(0,'/content/BiVLA/AutoGaze'); from autogaze.models.autogaze import AutoGaze, AutoGazeConfig; print('AutoGaze OK')"
conda run -n bivla python -c "import sys; sys.path.insert(0,'/content/BiVLA'); from shared_unified_policy import shared_task_policy_profile; print('controller OK')"

## 13. (선택) HuggingFace 로그인

`Yuqi1997/UniVLA`, `BAAI/*` 는 공개 repo라 보통 로그인 없이 받아져. 막히면 아래로 로그인.
**토큰을 평문으로 코드에 넣지 말 것** — 입력창(getpass) 사용.

In [ ]:
from huggingface_hub import login
from getpass import getpass
tok = getpass("HF token (없으면 그냥 Enter): ").strip()
if tok:
    login(tok); print("logged in")
else:
    print("로그인 건너뜀 (공개 repo면 OK)")

## 14. 모델 다운로드 → `/content/BiVLA/pretrain`

코드(`configs/paths.sh`)가 `BIVLA_ROOT/pretrain`에서 찾으므로 **레포 안 pretrain**으로 받는다.
- `Emu3-VisionTokenizer` (VQ-VAE)
- `UNIVLA_SIMPLER_BRIDGE_VIDEO_BS128_20K` (정책, ~14GB)
- fast tokenizer는 레포에 이미 번들 → 다운로드 불필요

In [ ]:
from huggingface_hub import snapshot_download
PRE = "/content/BiVLA/pretrain"

print("1/2 Emu3-VisionTokenizer ...")
snapshot_download("BAAI/Emu3-VisionTokenizer", local_dir=f"{PRE}/Emu3-VisionTokenizer")

print("2/2 UniVLA policy (UNIVLA_SIMPLER_BRIDGE_VIDEO_BS128_20K, ~14GB) ...")
snapshot_download("Yuqi1997/UniVLA",
                  allow_patterns="UNIVLA_SIMPLER_BRIDGE_VIDEO_BS128_20K/*",
                  local_dir=PRE)
print("done ->", PRE)

## 15. 토크나이저 보컬(`emu3.tiktoken`) 확보

Emu3 토크나이저는 정책 폴더 안에 `emu3.tiktoken`(BPE 보컬, 작은 파일)이 필요.
정책 ckpt에 없으면 `BAAI/Emu3-Stage1`에서 **그 파일 하나만** 받아 복사 (34GB 전체 불필요).

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

emu_hub = "/content/BiVLA/pretrain/UNIVLA_SIMPLER_BRIDGE_VIDEO_BS128_20K"
target = os.path.join(emu_hub, "emu3.tiktoken")
if os.path.exists(target):
    print("emu3.tiktoken 이미 존재 — OK")
else:
    print("emu3.tiktoken 없음 → BAAI/Emu3-Stage1 에서 가져옴 ...")
    p = hf_hub_download("BAAI/Emu3-Stage1", filename="emu3.tiktoken")
    shutil.copy(p, target)
    print("복사 완료 ->", target)

## 16. 경로 검증

In [ ]:
import os
EMU = "/content/BiVLA/pretrain/UNIVLA_SIMPLER_BRIDGE_VIDEO_BS128_20K"
VQ  = "/content/BiVLA/pretrain/Emu3-VisionTokenizer"
FAST= "/content/BiVLA/UniVLA/pretrain/fast_bridge_t5_s50"
for name, p, need in [
    ("policy", EMU, ["config.json","emu3.tiktoken"]),
    ("vq",     VQ,  ["config.json"]),
    ("fast",   FAST,["tokenizer.json"]),
]:
    ok = os.path.isdir(p)
    print(f"{'OK' if ok else 'MISSING'} {name}: {p}")
    if ok:
        files = set(os.listdir(p))
        for n in need:
            print(f"   {'OK' if n in files else 'MISSING'}: {n}")

## 16b. (중요) FAST 액션 토크나이저 패치

transformers가 `processor_config.json`의 `scale=50, min_token=-112`를 무시하고 코드 기본값(10, 0)을 써서 **액션 디코딩이 깨지는 버그**가 있다. 고치지 않으면 로봇 팔이 거의 안 움직여 success_rate=0%가 된다. 아래 셀로 기본값을 강제 교정한다.


In [ ]:
# FIX: action tokenizer config (scale=50, min_token=-112) gets dropped by transformers,
# falling back to wrong defaults (10,0) -> broken action decoding -> robot does not move.
import os, shutil
f = "/content/BiVLA/UniVLA/pretrain/fast_bridge_t5_s50/processing_action_tokenizer.py"
s = open(f).read()
s = s.replace("scale: float = 10", "scale: float = 50")
s = s.replace("min_token: int = 0", "min_token: int = -112")
open(f, "w").write(s)
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/modules/transformers_modules"), ignore_errors=True)
print("patched action tokenizer: scale=50, min_token=-112")


## 17. 평가 실행 스크립트 작성

`/content/run_univla.sh <task> <baseline|shared> <output_dir>`
- `baseline` = frozen UniVLA 그대로
- `shared`   = 멘토님 method (AutoGaze sparse + shared controller + layer pruning)
- `N_EPISODES` 환경변수로 에피소드 수 조절

In [ ]:
%%writefile /content/run_univla.sh
#!/bin/bash
set -e
TASK="$1"; MODE="$2"; OUTDIR="$3"

export MUJOCO_GL=osmesa
export VK_ICD_FILENAMES=/content/BiVLA/configs/nvidia_icd_egl.json
export UNIVLA_ROOT=/content/BiVLA/UniVLA
export SIMPLER_ENV_PATH=/content/BiVLA/SimplerEnv
export PYTHONPATH=/content/BiVLA:/content/BiVLA/AutoGaze:$PYTHONPATH

# headless display
Xvfb :99 -screen 0 1024x768x24 >/dev/null 2>&1 &
XVFB_PID=$!
sleep 2
export DISPLAY=:99

PY=/usr/local/envs/bivla/bin/python
COMMON=(
  "$PY" /content/BiVLA/adaptive_sparse_vla/eval.py
  --emu-hub  /content/BiVLA/pretrain/UNIVLA_SIMPLER_BRIDGE_VIDEO_BS128_20K
  --vq-hub   /content/BiVLA/pretrain/Emu3-VisionTokenizer
  --fast-path /content/BiVLA/UniVLA/pretrain/fast_bridge_t5_s50
  --task "$TASK"
  --n-episodes "${N_EPISODES:-24}"
  --output-dir "$OUTDIR"
  --image-size 256 --min-pixels 6400 --device cuda
  --save-video
)

if [ "$MODE" = "baseline" ]; then
  "${COMMON[@]}" --model-type baseline
else
  "${COMMON[@]}" --model-type shared_compact_focus \
    --patch-grid-size 20 --max-highres-patches 60 --lowres-factor 0.1 \
    --patch-grasp-steps 12 \
    --selector-hub nvidia/AutoGaze --selector-target-patch-size 16 \
    --selector-task-loss-requirement -1 --selector-device cuda \
    --selector-refresh-stride 4 --sparse-gate-mode auto \
    --llm-prune-count 2 --llm-prune-min-layer 0.75 --llm-prune-min-gap 1 \
    --uniform-gate-dx-ratio 0.20 --uniform-gate-dy-ratio 0.05 \
    --uniform-gate-area-min 0.010 --uniform-gate-area-max 0.0135 \
    --focus-gate-min-confidence 0.65 --focus-gate-max-confidence 0.78 \
    --focus-gate-min-context-confidence 0.55
fi

kill $XVFB_PID 2>/dev/null || true

## 18. 스모크 테스트 (baseline, 2 에피소드)

먼저 baseline 2 에피소드로 모델 로드 + 시뮬이 끝까지 도는지 확인. (Emu3 14GB 로드라 첫 셀은 몇 분 걸림)

In [ ]:
!N_EPISODES=2 bash /content/run_univla.sh widowx_put_eggplant_in_basket baseline /content/results/smoke_baseline

## 19. 전체 평가 — baseline (대조군)

4개 task × 24 에피소드. A100/L4 기준 task당 수십 분. 세션 끊기면 task별로 나눠 실행.

In [ ]:
import subprocess, os
TASKS = ["widowx_put_eggplant_in_basket","widowx_spoon_on_towel",
         "widowx_carrot_on_plate","widowx_stack_cube"]
for t in TASKS:
    print(f"\n===== baseline: {t} =====", flush=True)
    subprocess.run(["bash","/content/run_univla.sh",t,"baseline",f"/content/results/{t}_baseline"],
                   env={**os.environ,"N_EPISODES":"24"}, check=False)

## 20. 전체 평가 — shared (멘토님 method: AutoGaze + controller)

In [ ]:
import subprocess, os
TASKS = ["widowx_put_eggplant_in_basket","widowx_spoon_on_towel",
         "widowx_carrot_on_plate","widowx_stack_cube"]
for t in TASKS:
    print(f"\n===== shared: {t} =====", flush=True)
    subprocess.run(["bash","/content/run_univla.sh",t,"shared",f"/content/results/{t}_shared"],
                   env={**os.environ,"N_EPISODES":"24"}, check=False)

## 21. 결과 비교 (baseline vs shared)

In [ ]:
import json, os
TASKS = [("Eggplant","widowx_put_eggplant_in_basket"),("Spoon","widowx_spoon_on_towel"),
         ("Carrot","widowx_carrot_on_plate"),("Stack","widowx_stack_cube")]
def load(d,k):
    fp=os.path.join(d,f"results_{k}.json")
    return json.load(open(fp)) if os.path.exists(fp) else None
print(f"{'Task':<12}{'base succ':>10}{'shared succ':>12}{'d':>8}")
print("-"*42)
for label,k in TASKS:
    b=load(f"/content/results/{k}_baseline",k); s=load(f"/content/results/{k}_shared",k)
    bs=f"{b['success_rate']:.1%}" if b else "N/A"
    ss=f"{s['success_rate']:.1%}" if s else "N/A"
    d=f"{s['success_rate']-b['success_rate']:+.1%}" if (b and s) else ""
    print(f"{label:<12}{bs:>10}{ss:>12}{d:>8}")
print("-"*42)

---
## 참고

- **baseline vs shared**: shared가 멘토님 최종 method. README 7.1 기준 전체 82.29% → 84.38%
  (주로 carrot이 17/24 → 19/24). 큰 폭은 아님.
- **AutoGaze 자동 다운로드**: 첫 shared 실행 시 `nvidia/AutoGaze`가 받아짐. 네트워크 막히면 미리 받아둘 것.
- **flash-attn 불필요**: Emu3는 flash-attn 없으면 eager로 자동 fallback. Colab에서 빌드 안 해도 됨.
- **OOM**: Emu3 14GB + AutoGaze. A100(40GB) 여유, L4(24GB) 빠듯하면 baseline부터.
